# 🚀 Entrenamiento LSTM - Horizonte LONG

**GPU disponible:** Tesla T4 x2 (gratis)
**Tiempo estimado:** 60-90 minutos
**Costo:** $0

---

## ⚙️ IMPORTANTE: Activar GPU
1. Click derecho en este notebook → **Settings**
2. En **Accelerator**, selecciona **GPU T4 x2**
3. Click **Save**

---

## 1️⃣ Verificar GPU y configurar Multi-GPU

In [ ]:
import tensorflow as tf
import gc

# Forzar limpieza inicial
gc.collect()

print("="*80)
print("CONFIGURACIÓN DE KAGGLE - HORIZONTE LONG")
print("="*80)
print(f"TensorFlow version: {tf.__version__}")

# Detectar GPUs
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU disponible: {gpus}")

if len(gpus) > 0:
    print(f"\n✅ {len(gpus)} GPU(s) DETECTADA(S)")
    
    # Configurar memoria dinámica para cada GPU
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu.name}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(f"   Advertencia: {e}")
    
    # Configurar estrategia multi-GPU
    if len(gpus) > 1:
        strategy = tf.distribute.MirroredStrategy()
        print(f"\n🚀 ESTRATEGIA MULTI-GPU ACTIVADA")
        print(f"   Replicas: {strategy.num_replicas_in_sync}")
        print(f"   Velocidad esperada: {strategy.num_replicas_in_sync}x más rápido")
    else:
        strategy = tf.distribute.get_strategy()
        print("\n🚀 Usando 1 GPU")
else:
    print("\n⚠️ GPU NO DETECTADA - Ve a Settings → Accelerator → GPU T4 x2")
    strategy = tf.distribute.get_strategy()

print("="*80)

## 2️⃣ Instalar dependencias

In [ ]:
!pip install -q openpyxl seaborn
print("\n✅ Dependencias instaladas")

# Liberar memoria
import gc
gc.collect()

## 3️⃣ Configurar dataset

**Asegúrate de haber añadido tu dataset con:**
- Click **Add Data** → Selecciona tu dataset
- Debe contener: `train_all_customers_temporal.py` y `online_retail_2.xlsx`

In [ ]:
import os

# Listar datasets disponibles
print("📂 Datasets disponibles:")
!ls -lh /kaggle/input/

# IMPORTANTE: Ajusta el nombre de tu dataset aquí
DATASET_NAME = "lstm-customer-training"  # Cambia esto al nombre de tu dataset

print(f"\n📂 Contenido del dataset '{DATASET_NAME}':")
!ls -lh /kaggle/input/{DATASET_NAME}/

## 4️⃣ Crear estructura y copiar archivos

In [ ]:
import shutil
import gc

# Crear estructura de directorios (solo LONG)
!mkdir -p data/processed
!mkdir -p models/temporal/customer/long

# Copiar archivos desde el dataset (ajusta la ruta según tu dataset)
print("📥 Copiando archivos...")

# Ajusta el nombre del dataset aquí
DATASET_PATH = f"/kaggle/input/{DATASET_NAME}"

# Copiar dataset
!cp {DATASET_PATH}/online_retail_2.xlsx data/processed/

# Copiar script de entrenamiento
!cp {DATASET_PATH}/train_all_customers_temporal.py .

print("\n✅ Estructura creada")
print("\n📂 Verificación:")
!ls -lh data/processed/
!ls -lh *.py

# Liberar memoria
gc.collect()

## 5️⃣ Importar script de entrenamiento

In [ ]:
import sys
import gc

sys.path.append('.')

try:
    from train_all_customers_temporal import CustomerTemporalAnalyzer, TemporalConfig

    # PARCHE: Reducir batch_size para LONG (evitar Out of Memory)
    print("⚙️ Aplicando optimización de memoria para LONG...")
    TemporalConfig.LONG['batch_size'] = 16  # Reducido de 64 a 16

    print("✅ Script importado correctamente")
    print(f"✅ LONG batch_size optimizado: {TemporalConfig.LONG['batch_size']} (bajo uso de RAM)")
except Exception as e:
    print(f"❌ ERROR: {e}")
    raise

# Liberar memoria
gc.collect()

## 6️⃣ Configuración de entrenamiento

**Solo entrenaremos LONG (240 días → 60 días)**

In [ ]:
import gc

# Solo LONG
horizons_to_train = [
    TemporalConfig.LONG
]

print(f"✅ Configurado para entrenar {len(horizons_to_train)} horizonte(s): LONG")
print(f"\n📊 Configuración LONG:")
print(f"   Ventana: {TemporalConfig.LONG['window_days']} días")
print(f"   Pronóstico: {TemporalConfig.LONG['forecast_days']} días")
print(f"   Epochs: {TemporalConfig.LONG['epochs']}")
print(f"   Batch size: {TemporalConfig.LONG['batch_size']}")
print(f"   LSTM units: {TemporalConfig.LONG['lstm_units']}")

# Liberar memoria
gc.collect()

## 7️⃣ Inicializar y preparar datos (con optimización de memoria)

In [ ]:
import warnings
import gc
import numpy as np
warnings.filterwarnings('ignore')
from datetime import datetime

# Limpieza agresiva antes de empezar
gc.collect()

start_time = datetime.now()
print(f"⏰ Inicio: {start_time.strftime('%Y-%m-%d %H:%M:%S')}\n")

# Inicializar
print("="*70)
print("INICIALIZANDO ANALYZER")
print("="*70)

analyzer = CustomerTemporalAnalyzer(
    data_path='data/processed/online_retail_2.xlsx',
    output_dir='models/temporal/customer'
)

# FASE 1: Preparar datos con limpieza de memoria entre pasos
print("\n" + "="*70)
print("FASE 1: Preparación de datos")
print("="*70)

print("\n🔄 Paso 1/3: Cargando y preprocesando datos...")
analyzer.load_and_preprocess_data()
gc.collect()  # Liberar memoria
print("   ✅ Datos cargados")

print("\n🔄 Paso 2/3: Calculando métricas RFM...")
analyzer.calculate_rfm_metrics()
gc.collect()  # Liberar memoria
print("   ✅ Métricas RFM calculadas")

print("\n🔄 Paso 3/3: Generando secuencias temporales...")
analyzer.generate_customer_sequences(min_transactions=5)

# OPTIMIZACIÓN CRÍTICA: Reducir número de clientes para LONG
print("\n⚙️ OPTIMIZACIÓN DE MEMORIA PARA LONG:")
print(f"   Clientes totales: {len(analyzer.customers)}")

# Seleccionar solo los clientes con más compras (más datos de calidad)
analyzer.customers = sorted(analyzer.customers, 
                           key=lambda x: x['TotalPurchases'], 
                           reverse=True)[:1000]  # Solo top 1000 clientes

print(f"   Clientes seleccionados: {len(analyzer.customers)} (top por compras)")
print(f"   Promedio compras/cliente: {np.mean([c['TotalPurchases'] for c in analyzer.customers]):.1f}")

gc.collect()  # Liberar memoria
print("   ✅ Secuencias optimizadas")

print("\n" + "="*70)
print("✅ DATOS PREPARADOS - Memoria optimizada")
print("="*70)

## 8️⃣ ENTRENAR MODELO LONG

**Esta celda tomará 60-90 minutos con GPU T4 x2.**

Puedes cerrar la pestaña y volver después. Kaggle seguirá ejecutando mientras la sesión esté activa.

In [ ]:
import time
import gc

print(f"\n{'='*70}")
print(f"FASE 2: Entrenamiento de LONG")
print(f"{'='*70}")

results = {}

for i, horizon_config in enumerate(horizons_to_train, 1):
    print(f"\n\n{'█'*70}")
    print(f"HORIZONTE {i}/{len(horizons_to_train)}: {horizon_config['name'].upper()}")
    print(f"{'█'*70}")
    
    horizon_start = time.time()
    
    try:
        # Forzar limpieza de memoria antes de entrenar
        gc.collect()
        
        print("\n🚀 Iniciando entrenamiento...\n")
        model, history, metrics = analyzer.train_horizon_model(horizon_config)
        
        horizon_duration = (time.time() - horizon_start) / 60
        
        results[horizon_config['name']] = {
            'status': 'SUCCESS',
            'metrics': metrics,
            'duration_minutes': horizon_duration
        }
        
        print(f"\n✅ {horizon_config['name'].upper()} completado en {horizon_duration:.1f} minutos")
        
        # Liberar memoria del modelo
        del model
        del history
        gc.collect()
        
    except Exception as e:
        print(f"\n❌ Error entrenando {horizon_config['name']}: {e}")
        import traceback
        traceback.print_exc()
        results[horizon_config['name']] = {
            'status': 'FAILED',
            'error': str(e)
        }

# Resumen final
end_time = datetime.now()
total_duration = (end_time - start_time).total_seconds() / 60

print(f"\n\n{'═'*70}")
print("RESUMEN FINAL DE ENTRENAMIENTO")
print(f"{'═'*70}\n")

for horizon, result in results.items():
    status_icon = "✅" if result['status'] == 'SUCCESS' else "❌"
    print(f"{status_icon} {horizon.upper()}: {result['status']}")
    
    if result['status'] == 'SUCCESS':
        metrics = result['metrics']
        print(f"   Accuracy: {metrics['purchase_prob_accuracy']*100:.2f}%")
        print(f"   AUC: {metrics['purchase_prob_auc']:.4f}")
        print(f"   Days MAE: {metrics['days_mae']:.2f}")
        print(f"   Value MAE: ${metrics['value_mae']:.2f}")
        print(f"   Tiempo: {result['duration_minutes']:.1f} min")
    else:
        print(f"   Error: {result.get('error', 'Unknown')}")
    print()

print(f"⏰ Tiempo total: {total_duration:.1f} minutos ({total_duration/60:.1f} horas)")
print(f"\n💾 Modelos guardados en: {analyzer.output_dir}/")
print("\n" + "="*70)
print("✅ ENTRENAMIENTO COMPLETADO")
print("="*70)

## 9️⃣ Descargar modelo LONG

**Kaggle permite descargar archivos directamente desde el notebook.**

In [ ]:
# Comprimir modelo LONG
!cd models/temporal && zip -r customer_long_kaggle.zip customer/long/

print("✅ Modelo LONG comprimido")
print("\n📥 Para descargar:")
print("   1. Click en 'Output' (arriba a la derecha)")
print("   2. Descarga 'customer_long_kaggle.zip'")
print("   3. Descomprime en: E:\\Codigos\\Proyecto Final\\models\\temporal\\customer\\")

!ls -lh models/temporal/*.zip

## 🔟 Verificar archivos generados

In [ ]:
print("📊 ARCHIVOS GENERADOS PARA LONG:\n")
!ls -lh models/temporal/customer/long/